In [ ]:
import sys
import stardist
import csbdeep
import tensorflow as tf
import os
import numpy as np
import tifffile

print("Python:", sys.executable)
print("StarDist:", stardist.__version__)
print("CSBDeep:", csbdeep.__version__)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
from stardist.models import StarDist2D

model = StarDist2D(
    None,
    name="Model",
    basedir="/mnt/c/Users/hbruce/Desktop/stardist_cIN_model"
)

print("Model loaded successfully")

In [ ]:
#Provide input paths

input_path = "/mnt/c/Users/hbruce/Desktop/stardist_cIN_model/stardist_input/HB_E13_MGE_Sex_02032026_Embryo1.tif"

movie = tifffile.imread(input_path)

print("Movie shape:", movie.shape)
print("Data type:", movie.dtype)
print("Number of frames:", movie.shape[0])

In [17]:
with tifffile.TiffFile(input_path) as tif:

    # Read image
    movie = tif.asarray()

    # ImageJ metadata, if present
    imagej_metadata = tif.imagej_metadata

    if imagej_metadata is None:
        imagej_metadata = {}

    # Make a copy so we don't modify tifffile's object
    imagej_metadata = dict(imagej_metadata)

    # First TIFF page
    page0 = tif.pages[0]

   
    # TIFF resolution

    x_resolution = None
    y_resolution = None
    resolution_unit = None

    if "XResolution" in page0.tags:
        x_resolution = page0.tags["XResolution"].value

    if "YResolution" in page0.tags:
        y_resolution = page0.tags["YResolution"].value

    if "ResolutionUnit" in page0.tags:
        resolution_unit = page0.tags["ResolutionUnit"].value


# Automatically obtain ImageJ calibration

    spacing = imagej_metadata.get("spacing")
    unit = imagej_metadata.get("unit")
    frame_interval = imagej_metadata.get("finterval")

    # Number of frames
    n_frames = imagej_metadata.get("frames", movie.shape[0])

    # Existing frame labels
    labels = imagej_metadata.get("Labels")

    print("\n================================")
    print("INPUT TIFF")
    print("================================")

    print("File:", os.path.basename(input_path))
    print("Shape:", movie.shape)
    print("dtype:", movie.dtype)

    print("\nImageJ metadata:")
    for key, value in imagej_metadata.items():
        print(f"  {key}: {value}")

    print("\nTIFF resolution:")
    print("  X:", x_resolution)
    print("  Y:", y_resolution)
    print("  Unit:", resolution_unit)

    print("\nCalibration:")
    print("  XY unit:", unit)
    print("  Z spacing:", spacing)
    print("  Frame interval:", frame_interval)
    print("  Frames:", n_frames)




INPUT TIFF
File: HB_E13_MGE_Sex_02032026_Embryo1.tif
Shape: (10, 1044, 1032)
dtype: uint8

ImageJ metadata:
  ImageJ: 1.54p
  images: 10
  frames: 10
  unit: micron
  finterval: 479.9943542480469
  spacing: 1.5382778571428573
  loop: False
  Labels: ['t001_z1_c1.tif', 't002_z1_c1.tif', 't003_z1_c1.tif', 't004_z1_c1.tif', 't005_z1_c1.tif', 't006_z1_c1.tif', 't007_z1_c1.tif', 't008_z1_c1.tif', 't009_z1_c1.tif', 't010_z1_c1.tif']
  Properties: {'UniqueName': 'true'}

TIFF resolution:
  X: (2640000, 1000000)
  Y: (2640000, 1000000)
  Unit: RESUNIT.NONE

Calibration:
  XY unit: micron
  Z spacing: 1.5382778571428573
  Frame interval: 479.9943542480469
  Frames: 10


In [7]:
#Create stardist labels

all_labels = []
all_points = []

for t in range(movie.shape[0]):
    print(f"\nProcessing frame {t + 1}/{movie.shape[0]}...")

    frame = movie[t].astype(np.float32)

    labels, details = model.predict_instances(
        frame,
        n_tiles=(2, 2)
    )

    points = details["points"]

    all_labels.append(labels)
    all_points.append(points)

    print(f"  Detected objects: {len(points)}")

print("\n================================")
print("ALL FRAMES PROCESSED")
print("================================")

for t, points in enumerate(all_points):
    print(f"Frame {t}: {len(points)} objects")


Processing frame 1/10...


100%|██████████| 4/4 [00:00<00:00,  7.36it/s]


  Detected objects: 24

Processing frame 2/10...


100%|██████████| 4/4 [00:00<00:00,  7.26it/s]


  Detected objects: 28

Processing frame 3/10...


...

  Detected objects: 20

Processing frame 9/10...


100%|██████████| 4/4 [00:00<00:00,  8.49it/s]


  Detected objects: 25

Processing frame 10/10...


100%|██████████| 4/4 [00:00<00:00, 10.34it/s]

  Detected objects: 25

ALL FRAMES PROCESSED
Frame 0: 24 objects
Frame 1: 28 objects
Frame 2: 25 objects
Frame 3: 25 objects
Frame 4: 23 objects
Frame 5: 25 objects
Frame 6: 25 objects
Frame 7: 20 objects
Frame 8: 25 objects
Frame 9: 25 objects


In [20]:
#Create and save segmentation stacks while preserving original metadata

segmentation = np.stack(all_labels).astype(np.uint16)

print("\nSegmentation:")
print("  Shape:", segmentation.shape)
print("  dtype:", segmentation.dtype)

output_metadata = dict(imagej_metadata)

output_metadata["images"] = segmentation.shape[0]
output_metadata["frames"] = segmentation.shape[0]

if spacing is not None:
    output_metadata["spacing"] = spacing

if unit is not None:
    output_metadata["unit"] = unit

if frame_interval is not None:
    output_metadata["finterval"] = frame_interval

if labels is not None:
    output_metadata["Labels"] = labels

if "Properties" in imagej_metadata:
    output_metadata["Properties"] = imagej_metadata["Properties"]
basename = os.path.splitext(
    os.path.basename(input_path)
)[0]

output_path = os.path.join(
    output_dir,
    basename + "_stardist_segmentation.tif"
)


write_kwargs = {
    "imagej": True,
    "metadata": output_metadata,
}

# Preserve the original XY resolution when available
if (
    x_resolution is not None
    and y_resolution is not None
):
    write_kwargs["resolution"] = (
        x_resolution,
        y_resolution
    )

if resolution_unit is not None:
    write_kwargs["resolutionunit"] = resolution_unit


tifffile.imwrite(
    output_path,
    segmentation,
    **write_kwargs
)


print("\n================================")
print("SEGMENTATION SAVED")
print("================================")

print("Output:")
print(output_path)

print("Shape:", segmentation.shape)
print("dtype:", segmentation.dtype)


Segmentation:
  Shape: (10, 1044, 1032)
  dtype: uint16

SEGMENTATION SAVED
Output:
/mnt/c/Users/hbruce/Desktop/stardist_cIN_model/Output/HB_E13_MGE_Sex_02032026_Embryo1_stardist_segmentation.tif
Shape: (10, 1044, 1032)
dtype: uint16
